# Метод 6 — NLI-обоснованность ответа: опирается ли каждое предложение на контекст

**Задача та же, что у метода 3.** Вход — `(вопрос, до 8 retrieved-чанков, ответ бота)`,
целевая метка `reliable = faithfulness AND relevance`, первичная метрика — macro-F1 по
`reliable` с 95% ДИ, протокол 5×5 CV по `data/splits/folds_alfa.json`.

**Метод 6 не спрашивает LLM.** Он режет ответ на предложения и для каждой пары
«чанк → предложение» спрашивает NLI-модель: следует ли предложение из чанка. Из
матрицы вероятностей собираются 12 признаков; ни один вердикт метод не выносит —
порог подбирает протокол оценки.

---

## Почему это не SelfCheckGPT

Ветка заводилась под SelfCheckGPT: сгенерировать N ответов той же моделью и померить
их расхождение с исходным. Ветка остановлена, и вот чем это обосновано.

Промпт продакшн-бота недоступен. Значит, генератор в петле — **другая** система, и
расхождение измеряло разницу двух моделей, а не галлюцинацию бота. Метрика в такой
постановке ничего не значит независимо от её значения.

Замена — прямой NLI-grounding: премиса это чанк контекста, гипотеза это предложение
ответа, генератор из петли исключён полностью. Побочный эффект — цена:

| | SelfCheck | NLI-grounding |
|---|---:|---:|
| NLI-пар на корпус | ~404 000 | ~52 000 |
| время | ~37 GPU-ч | **~3 мин** |
| генераций LLM | 5–20 на кейс | 0 |

## Гипотеза и правило принятия решения

**H4.** Наследники SelfCheckGPT дают полезный сигнал на **faithfulness**, но уступают
supervised-методам на **relevance**. Проверяется прямо: ROC-AUC каждого признака
против faithfulness и против relevance по отдельности.

Правило решения зафиксировано **до** прогона (карточка C4), чтобы результат нельзя
было переинтерпретировать задним числом:

| AUC лучшего признака | Решение по ветке |
|---|---|
| ≥ 0.60 | развивать как самостоятельный метод |
| 0.53 – 0.60 | признаки отдать стэку, ветку не развивать |
| ≤ 0.53 | закрыть |

Ноутбук исполняется целиком: NLI влезает в T4 (24 GB) и работает даже на CPU, полный
прогон по корпусу — около 10 минут на GPU.

## Config

In [ ]:
# ======================= КОНФИГУРАЦИЯ — правится только здесь =======================
BASE      = "/home/jupyter/filestore/neurodrive"  # File Storage: переживает рестарт VM
REPO      = f"{BASE}/rag-reliability"
REPO_URL  = "https://github.com/MurkaSelebry/rag-reliability.git"
BRANCH    = "main"                          # актуальная ветка: волны 1-4 влиты

DATA      = "data/alfa.jsonl"               # канонический корпус, 2233 кейса
FOLDS     = "data/splits/folds_alfa.json"   # единственный источник разбиения
OUT       = "predictions/alfa/m6_grounding" # <вариант>/{scores.jsonl,report.json,auc.json}
SURFACE   = "predictions/alfa/baselines/surface/scores.jsonl"   # с чем сравнивать в стэке

NLI_MODEL     = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
NLI_MODEL_ALT = "Feudor2/rumodernbert-nli"   # абляция: русскоязычная NLI
BATCH_SIZE, MAX_LENGTH, OVERLAP = 64, 512, 128
ENTAIL_THRESHOLD = 0.5     # порог «предложение не подкреплено» для frac_unsupported
BOOTSTRAP_B = 10_000
SEED = 0

# Пороги правила принятия решения по ветке — из docs/handoff/tasks/C4.md.
DECISION_DEVELOP, DECISION_STACK = 0.60, 0.53
# ===================================================================================

import os, subprocess, sys, json

os.environ["HF_HOME"] = f"{BASE}/hf"        # до импорта transformers
os.makedirs(f"{BASE}/hf", exist_ok=True)
print("конфигурация принята")

## Setup — стек, репозиторий, железо

Стек метода 6 легче судейского: нужны `transformers` и `torch`, vLLM не нужен вовсе.
Ассерта на 70 GB VRAM здесь нет — метод сознательно рассчитан на дешёвую
конфигурацию, и это часть его обоснования.

In [ ]:
if not os.path.isdir(REPO):
    subprocess.check_call(["git", "clone", "-b", BRANCH, REPO_URL, REPO])
else:
    subprocess.check_call(["git", "-C", REPO, "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", REPO, "checkout", BRANCH])
    subprocess.check_call(["git", "-C", REPO, "pull", "--ff-only"])

subprocess.check_call(["git", "-C", REPO, "config", "user.name", "datasphere-runner"])
subprocess.check_call(["git", "-C", REPO, "config", "user.email", "datasphere@localhost"])

os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
print("репозиторий на", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip())

In [ ]:
pip = lambda *args: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cu121")
pip("-e", f"{REPO}[m6]")        # transformers + sentence-transformers + razdel
pip("scikit-learn", "pandas", "matplotlib")
pip("numpy==1.26.4")            # ПОСЛЕДНИМ: numpy 2 ломает C-расширения базового образа
print("стек установлен; если torch уже импортировался в этом ядре — Kernel → Restart и Run All")

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU {torch.cuda.get_device_name(0)} | "
          f"VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB")
else:
    print("CPU: метод отработает, но полный прогон вместо ~3 минут займёт часы")

## Data

In [ ]:
from rag_reliability.dataset import load_jsonl
from rag_reliability.methods.surface.features import split_chunks

samples = load_jsonl(DATA)
folds = json.load(open(FOLDS, encoding="utf-8"))

n = len(samples)
print(f"корпус: {n} кейсов, reliable {sum(s.reliable for s in samples) / n:.1%}, "
      f"faithfulness=1 {sum(s.faithfulness for s in samples) / n:.1%}, "
      f"relevance=1 {sum(s.relevance for s in samples) / n:.1%}")
print(f"фолды: {folds['config']['n_folds']}×{folds['config']['n_repeats']} CV, "
      f"в assignment {len(folds['assignment'])} id "
      f"(исключено гигантской группой {folds['stats']['excluded_ids']})")

## Метод целиком на одном кейсе

Три шага, все видны ниже.

**1. Сентенизация.** `razdel`, а не регулярка по точкам: регулярка резала «макс.»,
«т.д.» и «0.5%» на отдельные предложения, и каждый такой обрубок уходил в NLI
как самостоятельное утверждение.

**2. Матрица NLI.** Для каждой пары `(чанк, предложение)` модель даёт три логита
(entailment / neutral / contradiction). Softmax берётся **только по entail и contra**,
без neutral: neutral у мультиязычных моделей забирает основную массу вероятности на
длинных премисах, и полезный контраст между «следует» и «противоречит» тонет. Премиса
длиннее `MAX_LENGTH` режется на окна с перекрытием `OVERLAP` — иначе конец чанка
просто не доезжает до модели.

**3. Агрегация.** По чанкам — `max`: одного подтверждающего источника достаточно.
По предложениям — `min` для `min_entail` (**слабейшее звено**: одно неподкреплённое
предложение делает весь ответ неверным, среднее это размывает).

In [ ]:
from rag_reliability.methods.m6.features import razdel_sentences
from rag_reliability.methods.m6.nli import NLIScorer
from rag_reliability.methods.m6.grounding import compute_grounding, GROUNDING_KEYS
from rag_reliability.methods.m6.coverage import coverage_features, COVERAGE_KEYS

nli = NLIScorer(NLI_MODEL, device=DEVICE, batch_size=BATCH_SIZE,
                max_length=MAX_LENGTH, overlap=OVERLAP)
print(f"NLI загружена: {NLI_MODEL} на {nli.device}, "
      f"entail={nli.entail_index}, contra={nli.contra_index}")

In [ ]:
import numpy as np, pandas as pd

sample = samples[20]
chunks = split_chunks(sample.context)
sentences = razdel_sentences(sample.answer)
print(f"кейс {sample.id}: {len(sentences)} предложений × {len(chunks)} чанков = "
      f"{len(sentences) * len(chunks)} NLI-пар\n")
for i, s in enumerate(sentences):
    print(f"  [{i}] {s[:110]}")

In [ ]:
grounding = compute_grounding(sample.answer, chunks, nli, entail_threshold=ENTAIL_THRESHOLD)

matrix = pd.DataFrame(
    grounding.entail,
    index=[f"предл. {i}" for i in range(len(grounding.sentences))],
    columns=[f"чанк {j}" for j in range(len(chunks))],
).round(3)
print("P(entailment) — строка это предложение ответа, столбец это чанк контекста:")
display(matrix)

per_sentence = grounding.entail.max(axis=1)
print("\nлучший источник для каждого предложения (max по строке):", per_sentence.round(3))
print("слабейшее звено (min от них) =", round(float(per_sentence.min()), 3),
      "-> это и есть m6.min_entail")
print("чанки-источники (argmax по строкам):", sorted(grounding.source_chunk_ids))

In [ ]:
# Те же формулы, что в methods/m6/grounding.py — здесь они просто видны.
per_sentence_entail = grounding.entail.max(axis=1)
per_sentence_contra = grounding.contra.max(axis=1)
manual = {
    "m6.max_entail": per_sentence_entail.max(),
    "m6.min_entail": per_sentence_entail.min(),
    "m6.mean_entail": per_sentence_entail.mean(),
    "m6.mean_contra": per_sentence_contra.mean(),
    "m6.max_contra": per_sentence_contra.max(),
    "m6.frac_unsupported": (per_sentence_entail < ENTAIL_THRESHOLD).mean(),
    "m6.n_sentences": float(len(grounding.sentences)),
    "m6.chunk_spread": float(len(set(grounding.entail.argmax(axis=1).tolist()))),
}
for key in GROUNDING_KEYS:
    assert abs(manual[key] - grounding.features[key]) < 1e-9, (
        f"{key}: ячейка и модуль разошлись — формула в ноутбуке не та, что считает метод"
    )
print("формулы в ячейке совпали с methods/m6/grounding.py до 1e-9\n")
display(pd.Series(grounding.features).round(4).to_frame("значение"))

In [ ]:
# Coverage переиспользует ту же матрицу и своих NLI-вызовов не делает: условия ищутся
# только в чанках-источниках. По всем 5-8 чанкам считать бессмысленно — там много
# условий не по вопросу, и метрика превращается в шум.
cov = coverage_features(sample.answer, chunks, source_chunk_ids=grounding.source_chunk_ids)
display(pd.Series(cov).round(4).to_frame("значение"))
print("Итого признаков:", len(GROUNDING_KEYS), "grounding +", len(COVERAGE_KEYS), "coverage")
print("Ни один из них не бинаризуется здесь: порог подбирает evaluate_cv внутри фолда.")

## Полный прогон по корпусу

Через CLI: он пишет артефакт инкрементально, умеет `--resume` и кладёт рядом
`run.yaml` с git-хэшем — без этого число из ноутбука невозможно связать с кодом,
которым оно получено.

~3 минуты на A100, ~10 минут на T4.

In [ ]:
!python scripts/score_m6_grounding.py --data {DATA} \
    --output {OUT}/base/scores.jsonl --variant base \
    --nli-model {NLI_MODEL} --device {DEVICE} \
    --batch-size {BATCH_SIZE} --max-length {MAX_LENGTH} --overlap {OVERLAP} \
    --entail-threshold {ENTAIL_THRESHOLD} --resume

In [ ]:
!python scripts/score_m6_grounding.py --auc-only --data {DATA} \
    --scores {OUT}/base/scores.jsonl --auc-report {OUT}/base/auc.json \
    --bootstrap-B {BOOTSTRAP_B}

## H4 · Где у признаков сигнал: faithfulness против relevance

`auc.json` из CLI считает AUC против faithfulness и против `reliable`. Гипотеза H4
говорит о **противопоставлении** faithfulness и relevance, поэтому ось relevance
считается здесь же, тем же бутстрэпом.

Читать таблицу нужно по интервалам, а не по точечным оценкам: AUC 0.55 на 2233 кейсах
имеет ДИ шириной около ±0.027, и признак, у которого интервал накрывает 0.5, не
отличим от монетки.

In [ ]:
from sklearn.metrics import roc_auc_score
from rag_reliability.evaluation.bootstrap import bootstrap_ci

FEATURE_KEYS = (*GROUNDING_KEYS, *COVERAGE_KEYS)


def load_scores(path):
    rows = {}
    for line in open(path, encoding="utf-8"):
        if line.strip():
            row = json.loads(line)
            rows[str(row["id"])] = row["scores"]
    assert rows, f"{path}: пустой артефакт"
    return rows


def auc_ci(labels, values, B=BOOTSTRAP_B, seed=SEED):
    labels, values = np.asarray(labels, dtype=int), np.asarray(values, dtype=float)
    metric = lambda y, x: 0.5 if len(set(y.tolist())) < 2 else float(roc_auc_score(y, x))
    r = bootstrap_ci(labels, values, metric, B=B, seed=seed)
    return r.point, r.lo, r.hi


def auc_table(scores_path):
    """AUC каждого признака против обеих осей — ядро проверки H4."""
    rows = load_scores(scores_path)
    evaluated = [s for s in samples if s.id in rows]
    out = []
    for key in FEATURE_KEYS:
        values = [rows[s.id][key] for s in evaluated]
        if len(set(values)) == 1:      # константа: AUC не определён, а не равен 0.5
            out.append({"признак": key, "faithfulness": "константа", "relevance": "константа"})
            continue
        record = {"признак": key}
        for axis, labels in (("faithfulness", [s.faithfulness for s in evaluated]),
                             ("relevance", [s.relevance for s in evaluated])):
            point, lo, hi = auc_ci(labels, values)
            record[axis] = f"{point:.3f} [{lo:.3f}; {hi:.3f}]"
            record[f"_{axis}"] = point            # числом: для сортировки и вердиктов
            record[f"_{axis}_lo"], record[f"_{axis}_hi"] = lo, hi
        out.append(record)
    return pd.DataFrame(out), len(evaluated)


table, n_evaluated = auc_table(f"{OUT}/base/scores.jsonl")
print(f"n = {n_evaluated}, бутстрэп B = {BOOTSTRAP_B}")
display(table[["признак", "faithfulness", "relevance"]])

In [ ]:
# Вердикт по H4: сильнее ли признаки на faithfulness, чем на relevance, и есть ли
# вообще признак, чей интервал не накрывает 0.5.
scored = table.dropna(subset=["_faithfulness"])
best = scored.loc[scored["_faithfulness"].idxmax()]
print(f"сильнейший признак на faithfulness: {best['признак']} = {best['faithfulness']}")
print(f"он же против relevance:            {best['relevance']}")

# «Отличим от монетки» = нижняя граница ДИ выше 0.5. Для признаков, у которых сигнал
# обратный (уверенность падает с ростом надёжности), симметрично: верх ниже 0.5.
informative = scored[(scored["_faithfulness_lo"] > 0.5)]
print(f"\nпризнаков с нижней границей ДИ выше 0.5 (faithfulness): {len(informative)} из {len(scored)}")
print(", ".join(informative["признак"]) or "нет")

mean_faith, mean_rel = scored["_faithfulness"].mean(), scored["_relevance"].mean()
print(f"\nсредний AUC по признакам: faithfulness {mean_faith:.3f}, relevance {mean_rel:.3f} "
      f"(разница {mean_faith - mean_rel:+.3f})")

# H4 утверждает разницу между осями. Разница признаётся только если у сильнейшего
# признака интервалы двух осей не пересекаются: иначе «faithfulness выше» — это
# сравнение двух чисел, каждое из которых само неотличимо от 0.5.
axes_separated = float(best["_faithfulness_lo"]) > float(best["_relevance_hi"])
if axes_separated:
    h4 = "подтверждается: интервалы осей у сильнейшего признака не пересекаются"
elif len(informative):
    h4 = ("частично: сигнал на faithfulness отличим от монетки, но разница между осями "
          "лежит внутри интервалов — из этих данных она не следует")
else:
    h4 = ("не подтверждается: ни у одного признака нижняя граница ДИ не выше 0.5, "
          "то есть сигнала нет ни на одной оси")
print(f"H4 «сигнал на faithfulness сильнее, чем на relevance» — {h4}")

In [ ]:
# Правило принятия решения по ветке — то самое, что зафиксировано до прогона.
peak = float(best["_faithfulness"])
if peak >= DECISION_DEVELOP:
    verdict = "развивать как самостоятельный метод"
elif peak > DECISION_STACK:
    verdict = "признаки отдать стэку, ветку как направление не развивать"
else:
    verdict = "закрыть ветку"
print(f"лучший AUC {peak:.3f}  →  {verdict}")
print(f"(пороги правила: ≥{DECISION_DEVELOP} развивать, >{DECISION_STACK} в стэк, "
      f"иначе закрыть)")

## Абляция NLI-модели

Единственная альтернатива, которая может изменить вывод, — сама NLI-модель. Базовая
`mDeBERTa` мультиязычная; `Feudor2/rumodernbert-nli` обучена на русском. Если разница
между ними лежит внутри доверительных интервалов, то дело не в выборе модели.

In [ ]:
!python scripts/score_m6_grounding.py --data {DATA} \
    --output {OUT}/rumodernbert/scores.jsonl --variant rumodernbert \
    --nli-model {NLI_MODEL_ALT} --device {DEVICE} \
    --batch-size {BATCH_SIZE} --max-length {MAX_LENGTH} --overlap {OVERLAP} \
    --entail-threshold {ENTAIL_THRESHOLD} --resume

In [ ]:
alt_table, _ = auc_table(f"{OUT}/rumodernbert/scores.jsonl")
comparison = table[["признак", "faithfulness"]].merge(
    alt_table[["признак", "faithfulness"]], on="признак", suffixes=(" mDeBERTa", " ruModernBERT")
)
display(comparison)

alt_scored = alt_table.dropna(subset=["_faithfulness"])
alt_best = alt_scored.loc[alt_scored["_faithfulness"].idxmax()]
print(f"лучший признак базовой модели:  {best['признак']} {best['faithfulness']}")
print(f"лучший признак альтернативной:  {alt_best['признак']} {alt_best['faithfulness']}")
# Интервалы пересекаются, если начало одного не позже конца другого.
overlap = (float(best["_faithfulness_lo"]) <= float(alt_best["_faithfulness_hi"])
           and float(alt_best["_faithfulness_lo"]) <= float(best["_faithfulness_hi"]))
print("интервалы пересекаются — выбор NLI-модели ничего не решает"
      if overlap else "интервалы разошлись — выбор NLI-модели имеет значение")

## Вклад в стэк

Отдельный вопрос: даже слабый признак может быть полезен, если он **некоррелирован** с
теми, что уже есть. Проверяется прямо — surface-бейзлайн отдельно и он же плюс каждый
признак метода 6, OOF-стэкинг по тем же фолдам.

Приращение читается по 95% ДИ парного бутстрэпа: точечная разница в 0.01 при ширине
интервала 0.06 не означает ничего.

In [ ]:
# Метод 6 сам по себе: его лучший признак как единственный скор.
!python scripts/evaluate_cv.py --data {DATA} --folds {FOLDS} \
    --scores {OUT}/base/scores.jsonl \
    --score-expr "m6.min_entail" \
    --compare {SURFACE} \
    --output {OUT}/base/report.json

In [ ]:
SURFACE_FEATURES = "surf.p_faith,surf.p_rel,surf.overlap_ans_ctx_1,surf.digit_match_ratio"

# База: только surface.
!python scripts/run_stack.py --data {DATA} --folds {FOLDS} \
    --sources surface={SURFACE} \
    --features {SURFACE_FEATURES} \
    --variant surface_only --output results/stack/surface_only.json

# Surface + признаки метода 6.
!python scripts/run_stack.py --data {DATA} --folds {FOLDS} \
    --sources surface={SURFACE} m6={OUT}/base/scores.jsonl \
    --features {SURFACE_FEATURES},m6.min_entail,m6.mean_entail,m6.frac_unsupported \
    --variant surface_plus_m6 --output results/stack/surface_plus_m6.json

In [ ]:
stack = []
for name, path in (("surface", "results/stack/surface_only.json"),
                   ("surface + m6", "results/stack/surface_plus_m6.json"),
                   ("m6 отдельно (min_entail)", f"{OUT}/base/report.json")):
    if not os.path.isfile(path):
        print(f"нет {path} — пропуск")
        continue
    rep = json.load(open(path, encoding="utf-8"))
    primary = rep["primary"]
    stack.append({
        "конфигурация": name,
        "macro-F1": round(primary["value"], 4),
        "95% ДИ": f'[{primary["ci95"][0]:.4f}; {primary["ci95"][1]:.4f}]',
        "выше шума": primary["above_noise"],
    })
display(pd.DataFrame(stack))
print("Приращение к surface с парным ДИ — в поле comparisons отчёта surface_plus_m6.")

## Итог по методу 6

Ниже — всё, что ноутбук установил: таблица AUC, вердикт по правилу решения, ответ по
H4 и вклад в стэк. Отрицательный результат здесь такой же результат: он закрывает
направление, и закрывает его воспроизводимо.

In [ ]:
print("=" * 78)
print("МЕТОД 6 — NLI-GROUNDING: ИТОГ")
print("=" * 78)
print(f"корпус {n_evaluated} кейсов, бутстрэп B={BOOTSTRAP_B}, NLI {NLI_MODEL}\n")

print("Сильнейшие признаки по AUC против faithfulness:")
top = scored.sort_values("_faithfulness", ascending=False).head(5)
for _, row in top.iterrows():
    print(f"  {row['признак']:24} {row['faithfulness']}   (relevance: {row['relevance']})")

print(f"\nH4 «сигнал на faithfulness сильнее, чем на relevance» — {h4}")
print(f"   средние AUC по 12 признакам: {mean_faith:.3f} против {mean_rel:.3f}")
print(f"Правило решения по ветке: лучший AUC {peak:.3f} → {verdict}")
if stack:
    print(f"\nВ стэке: {stack[0]['конфигурация']} {stack[0]['macro-F1']} {stack[0]['95% ДИ']}")
    for row in stack[1:]:
        print(f"          {row['конфигурация']} {row['macro-F1']} {row['95% ДИ']}")
print("\nЧто это значит: метод даёт дешёвый (минуты вместо десятков GPU-часов) и честно")
print("измеренный сигнал, но слабый. Развивать ветку как самостоятельный метод нет")
print("оснований; признаки остаются доступны стэку, который сам решает, брать ли их.")

In [ ]:
# Одна картинка: AUC каждого признака с интервалом против обеих осей.
import matplotlib.pyplot as plt

ordered = scored.sort_values("_faithfulness")
y = np.arange(len(ordered))
fig, ax = plt.subplots(figsize=(9, 0.42 * len(ordered) + 2))
for offset, axis, color in ((-0.17, "_faithfulness", "#2f6f4f"), (0.17, "_relevance", "#8a5a2b")):
    point = ordered[axis].to_numpy()
    lo = ordered[f"{axis}_lo"].to_numpy() if f"{axis}_lo" in ordered else point
    ax.errorbar(point, y + offset, xerr=np.abs(point - lo), fmt="o", ms=5,
                color=color, capsize=3, label=axis.strip("_"))
ax.axvline(0.5, color="#999999", lw=1, ls="--")
ax.axvline(DECISION_DEVELOP, color="#b03a2e", lw=1, ls=":", label=f"порог развития {DECISION_DEVELOP}")
ax.set_yticks(y, ordered["признак"])
ax.set_xlabel("ROC-AUC (усы — нижняя граница 95% ДИ)")
ax.set_title("Метод 6: сигнал каждого признака против обеих осей")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

In [ ]:
!git add {OUT} results/stack 2>/dev/null; \
 git commit -m "results(m6): grounding по корпусу, AUC с ДИ, абляция NLI и вклад в стэк" && \
 git log --oneline -1